# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [9]:
from dash.dcc import Markdown
from gradio.cli.commands import display

# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [10]:
# set up environment

from openai import OpenAI
from dotenv import load_dotenv
import json
import gradio as gr

load_dotenv(override=True)

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [11]:
system_message = """
You are a battle-hardened senior software developer who has expertise in all things Python and are able to explain in plain English the who, what, where and why behind a piece of code. When explaining code to others, you don the persona of a professor teaching their students.
"""

def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [12]:
explain_code_function = {
    "name": "explain_plain_code",
    "description": "Explain the purpose of code in a clear, concise manner to the responder.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The code that the requestor wants explained",
            },
        },
        "required": ["code"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": explain_code_function}]

def handle_code_function(message):
    responses = []
    codes = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "explain_plain_code":
            arguments = json.loads(tool_call.function.arguments)
            code = arguments.get('code')
            codes.append(code)
            code_details = codes[0]
            responses.append({
                "role": "tool",
                "content": code_details,
                "tool_call_id": tool_call.id
            })
    return responses, codes


def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, code = handle_code_function(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    return history, voice



In [13]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]  
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.
